# Two trainable baselines for wave classification

by Andrés Muñoz-Jaramillo (template), adapted for wave classification by Diego

This notebook defines two simple, trainable baselines for the EUV wave classification
task, to be evaluated on the validation set before fine-tuning the full Surya backbone:

1. **Constant-probability baseline** — a single learned scalar that predicts the class
   base rate (wave vs. no wave), ignoring the image entirely.
2. **Running-difference baseline** — AIA193(now) - AIA193(one cadence step earlier),
   mean-pooled 32x32, then a logistic regression.

It focuses on the concept of defining a PyTorch model, a PyTorch lightning training loop
and the definition of metrics of performance.

This notebook assumes familiarity with the concepts of datasets and dataloaders contained
in **_0_dataset_dataloader_template_diego.ipynb_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
from pathlib import Path
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# The wandb module itself, not just Lightning's WandbLogger wrapper. Two things below need
# it: ending one baseline's run before the next WandbLogger is built (otherwise Lightning
# reuses the live run and both baselines land on one set of axes), and attaching the
# per-epoch figures as run media.
import wandb

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks

torch.set_float32_matmul_precision('medium')


## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [5]:
# The config is the single source of truth. load_wave_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.test.configs import load_wave_config

cfg = load_wave_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: wave_classification


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [6]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# The linear baseline needs no backbone, so skip the 1.8 GB weights.
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


## Define Downstream (DS) dataset

This child class (`waveDSDataset`) takes as input all expected HelioFM parameters, plus
additional parameters relevant to the downstream application. Here we focus in particular
on the DS index (the wave/no-wave catalog) and the parameters necessary to combine it with
the HelioFM index.

The label is the binary `class` column (`"wave"` / `"no wave"`), encoded to `1.0` / `0.0`
by `wave_common.wave_label_transform`.

The train / validation / test datasets are built by **one** helper,
`experiments/wave_common.py:build_wave_datasets()`, which is also what
`2_finetune_template_1D_diego.ipynb` and `4_finetune_wave_1D.py` call. That is deliberate:
the baseline in this notebook and the Surya fine-tune in notebook 2 must be trained and
scored on the *same events*, or the comparison between them means nothing. Sharing the
construction is what guarantees it, rather than two cells that happen to agree today.


In [7]:
from downstream_apps.test.datasets.wave_dataset import waveDSDataset

In [29]:
# ---------------------------------------------------------------------------
# Datasets and dataloaders. One definition, shared with 2_finetune_template_1D_diego.ipynb
# and 4_finetune_wave_1D.py, so task 1 and task 2 train on the SAME events.
#
# This cell used to call build_helio_dataloaders() directly. That is the right entry point
# for a normal run, but for this task it carries four defects — and the same four reached
# notebook 2 and the training script, which is why the fix lives in wave_common.py rather
# than being patched here:
#
#   1. max_number_of_samples=cfg.data.max_samples head-slices the index by ds_index, so
#      max_samples=10 returned the 5 EARLIEST event pairs (all June-July 2010) — a
#      chronological sample, not a random one. event_stratified_subset() replaces it with a
#      seeded, EVENT-level draw: both samples of a chosen event are kept, so the result is
#      exactly 50/50 and accuracy 0.5 is exactly chance. Subsets are nested across N.
#   2. drop_last=True was applied to the VALIDATION loader too, silently discarding a
#      sample from the very metric ModelCheckpoint selects on. Now False for val/test.
#   3. prefetch_factor was not exposed, and it is the only bound on worker RAM:
#      worst case is num_workers * prefetch_factor * batch_size * (0.13 GB at 1 channel,
#      1.74 GB at 13). Left unbounded this machine gets OOM-killed.
#   4. label_transform was an inline lambda, which cannot be pickled, so any
#      num_workers > 0 died on spawn. wave_common.wave_label_transform is module-level.
#
# build_wave_datasets() also adds the held-out TEST split, and reads each S3 object into
# RAM rather than through the configured cache directory (that directory is on EFS, which
# measures 9 MB/s against 400 MB/s for the in-memory path).
# ---------------------------------------------------------------------------
import importlib

from downstream_apps.test.experiments import wave_common as wc

# Edits to wave_common.py do NOT reach a kernel that already imported it: the module object
# is cached in sys.modules, so re-running the import statement just rebinds the same stale
# object. That is why this cell raised
#   TypeError: build_wave_loader() got an unexpected keyword argument 'persistent_workers'
# even though the parameter was already in the file. reload() re-executes the module source.
#
# Safe here because nothing that survives this cell points into the old module: the datasets,
# the loaders and the label transform are all rebuilt below from the reloaded one, and
# finish_run() looks up wc.plot_run at call time rather than holding a reference.
# (Restarting the kernel and running top-to-bottom is the alternative, and is the only
# option if you ever change a CLASS in wave_common.py rather than a function.)
wc = importlib.reload(wc)

# Both baselines below read AIA193 and nothing else, but the dataset decodes every channel
# named in cfg.data.channels: 15.2 s/sample for all 13 vs 1.2 s/sample for AIA193 alone
# (measured on this machine). Narrowing it changes only which NetCDF variables are decoded
# — same index files, same events, same split, same seed — so the comparison against Surya
# in notebook 2 (which keeps all 13, because the backbone needs them) is still over
# identical events. This is what makes a full-size run finish inside a notebook.
# Set to False to load all 13 and get byte-identical tensors instead.
BASELINE_CHANNELS_ONLY = True
if BASELINE_CHANNELS_ONLY:
    cfg.data.channels = ["aia193"]

TRAIN_N = 384        # samples (192 events). None keeps every complete pair (~1210).
SUBSET_SEED = 42     # fixes WHICH events; shared with notebook 2 and the training script
NUM_WORKERS = 4
PREFETCH_FACTOR = 1

# ---------------------------------------------------------------------------
# PERSISTENT_WORKERS = False is what makes this notebook re-runnable, and it is the fix for
# the "RuntimeError: Please call `iter(combined_loader)` first." raised by the
# running-difference fit below. That message is a red herring. The real chain was:
#
#   * persistent_workers=True keeps the spawned worker pool alive after the iterator is
#     exhausted, which makes the DataLoader STATEFUL: torch's DataLoader.__iter__ then takes
#     its `self._iterator._reset(self)` branch instead of building a fresh pool.
#   * The constant-probability fit above was interrupted (its output shows "Detected
#     KeyboardInterrupt"), and Lightning's graceful shutdown reaped those four workers.
#   * The next fit reused the SAME loader objects, so _reset() sent its resume handshake to
#     dead PIDs, waited, got a queue.Empty, and raised the ACTUAL error:
#     "DataLoader worker (pid(s) 2374, 2392, 2410, 2428) exited unexpectedly".
#   * Lightning caught that inside fit_loop.setup_data() and tore the loop down. Teardown
#     calls _DataFetcher.reset(), which needs len(combined_loader) — but iter() had never
#     succeeded, so CombinedLoader._iterator was still None, and the misleading
#     "Please call iter(combined_loader) first." surfaced on top of the real failure.
#
# With persistent_workers=False every iter() builds a fresh pool, so the loaders survive
# interrupts, cell re-runs and back-to-back fits. The cost is re-paying spawn's re-import
# once per epoch (a few seconds for 4 workers), negligible against the data pass itself.
#
# Do NOT "fix" this by setting NUM_WORKERS = 0. The workers are not the problem: the
# batch-inspection cell below already pulled (2, 1, 2, 4096, 4096) through these same four
# spawn workers, so spawn works fine under this kernel. And nothing is cached between
# epochs — every sample re-fetches ~1.18 GB from S3, which only reaches full throughput
# with several readers in parallel, so num_workers=0 would make a 384-sample epoch
# unusably slow.
# ---------------------------------------------------------------------------
PERSISTENT_WORKERS = False

train_dataset, val_dataset, test_dataset = wc.build_wave_datasets(
    cfg, scalers, include_test=True, in_memory=True,
)

# Subset in place, and only AFTER construction: the Surya-index match has to have happened
# before events can be counted, because only events whose *both* samples survived the
# match are eligible — that is what makes the result exactly class-balanced.
# val and test take n_samples=None: every complete pair, nothing thrown away.
for split_name, dataset, n in [("train", train_dataset, TRAIN_N),
                               ("val", val_dataset, None),
                               ("test", test_dataset, None)]:
    info = wc.event_stratified_subset(dataset, n_samples=n, seed=SUBSET_SEED)
    print(f"{split_name:>5}: {info.describe()}")

# shuffle=True only for train; drop_last=True only for train (a partial batch adds noise to
# the gradient, but dropping val samples corrupts the reported metric).
train_data_loader = wc.build_wave_loader(
    train_dataset, batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, seed=SUBSET_SEED, shuffle=True, drop_last=True,
    persistent_workers=PERSISTENT_WORKERS,
)
val_data_loader = wc.build_wave_loader(
    val_dataset, batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, seed=SUBSET_SEED, shuffle=False, drop_last=False,
    persistent_workers=PERSISTENT_WORKERS,
)

batch_size = cfg.batch_size
worst_case_gb = NUM_WORKERS * PREFETCH_FACTOR * batch_size * 0.134 * len(cfg.data.channels)
print(f"\nchannels: {cfg.data.channels}")
print(f"train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)} "
      f"samples | batch_size: {batch_size}")
print(f"worst-case loader RAM: {worst_case_gb:.1f} GB "
      f"({NUM_WORKERS} workers x prefetch {PREFETCH_FACTOR} x batch {batch_size})")
print(f"persistent_workers: {PERSISTENT_WORKERS} "
      f"-> loaders are safe to reuse across fits and interrupts")

train: 384 samples / 192 events (192 wave, 192 no wave) from 596 complete pairs available; dropped 18 unpaired sample(s)
  val: 24 samples / 12 events (12 wave, 12 no wave) from 12 complete pairs available; dropped 1 unpaired sample(s)
 test: 48 samples / 24 events (24 wave, 24 no wave) from 24 complete pairs available; dropped 2 unpaired sample(s)

channels: ['aia193']
train: 384 | val: 24 | test: 48 samples | batch_size: 2
worst-case loader RAM: 1.1 GB (4 workers x prefetch 1 x batch 2)
persistent_workers: False -> loaders are safe to reuse across fits and interrupts


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [9]:
print("kernel alive check")

kernel alive check


In [10]:
# Inspect a single batch to confirm shapes before training. Dimension mismatches are the
# dominant source of error in this kind of work, so this is worth a cell of its own.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})

# Derive model dimensions from the batch/config rather than hardcoding them.
#
# aia193_index MUST come from cfg.data.channels, never be written as a literal: it is 3 in
# the 13-channel configuration and 0 once the cell above narrows channels to ["aia193"].
# A hardcoded 3 would silently read a non-existent channel (IndexError) or, in a 13-channel
# run with a reordered list, train the baseline on the wrong wavelength.
n_channels = len(cfg.data.channels)
aia193_index = cfg.data.channels.index("aia193")
img_size = batch["ts"].shape[-1]     # reported only; the model no longer needs it
print(f"n_channels={n_channels} | aia193_index={aia193_index} | img_size={img_size}")

# ts is (B, C, T, H, W). T must be 2: the running difference is frame[-1] - frame[0], so a
# single timestep would make the feature identically zero.
assert batch["ts"].shape[2] == 2, (
    f"expected 2 input timesteps, got {batch['ts'].shape[2]} — check "
    f"data.time_delta_input_minutes and model.time_embedding.time_dim in the config"
)


{'ts': (2, 1, 2, 4096, 4096), 'time_delta_input': (2, 2), 'forecast': (2,), 'ds_index': 'list'}
n_channels=1 | aia193_index=0 | img_size=4096


## Define simple baseline models

Defining a simple baseline is important to understand what value is bringing the AI model
to the problem.

It is always very good to have a very simple baseline model. Ideally one that cannot
overfit the data. This is a very good way of really measuring the value added of complex
models. Classical machine learning excels here:

- Regressions and logistic regressions.
- Climatological averages.
- Persistance.
- Simple transformations.

Simple models avoid excesively optimistic assessments of the capatiblities of complex
models and for many problems are actually remarkably hard to beat.

Here we define two such baselines:

1. `ConstantProbabilityModel` — a single trainable scalar (logit) that ignores the input
   entirely and learns the wave/no-wave base rate.
2. `RunningDifferenceLogisticModel` — AIA193(now) - AIA193(one cadence step earlier),
   mean-pooled with a 32x32 kernel, then a linear layer (logistic regression via
   `BCEWithLogitsLoss`).

As with the dataset, we import both from a module so they can be reused in training
scripts later on.

In [11]:
from downstream_apps.test.models.simple_baseline import (
    ConstantProbabilityModel,
    RunningDifferenceLogisticModel,
    destandardize_channels,
    # The feature the running-difference baseline is built on, exposed as a plain function
    # so the diagnostic cell at the end of this notebook can score the validation set
    # directly — without it, a 1-feature model's result is impossible to interpret.
    mean_abs_pooled_running_difference,
)


We can now test that both models manipulate a batch as expected and return a wave
probability logit.

`ConstantProbabilityModel` needs no configuration — it ignores the image, and has exactly
**one** parameter. `RunningDifferenceLogisticModel` needs only the AIA193 channel index
(pulled from the config, never hardcoded) and the pooling kernel; it has exactly **two**
parameters, a weight and a bias on a single scalar feature.

That parameter count is the point of the whole notebook. A two-parameter model that is
monotone in one feature is a *threshold detector*: it cannot memorize the training set, and
its AUROC is a fixed property of the feature rather than something training discovers. Only
the loss and the calibration are learned. So if Surya beats this, the gain is real.

Note that since neither model has been trained yet and was initialized randomly, the
output here has no real meaning. It only acts as a test that the forward pass doesn't have
dimension problems — dimension problems are the dominant source of error in this kind of
work.


In [12]:
POOL_KERNEL = 32   # mean-pool kernel for the running difference; 4096 -> 128x128

# Baseline 1: a single learned logit, independent of the input.
model_constant = ConstantProbabilityModel()

# Baseline 2: AIA193 running difference, mean-pooled, magnitude-averaged, then a logistic
# regression on that one scalar. It expects 'ts' in signum-log space — destandardize_channels()
# is wired in as the Lightning module's preprocess_fn below, so forward() always sees it.
#
# NOTE: no img_size argument. The model pools with a fixed kernel and then averages over
# whatever spatial extent remains, so it is resolution-independent by construction and the
# frame size is not something it needs to be told.
model_running_diff = RunningDifferenceLogisticModel(
    channel_index=aia193_index,
    pool_kernel=POOL_KERNEL,
)

for name, m in [("constant", model_constant), ("running_diff", model_running_diff)]:
    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:>13}: {n_trainable} trainable parameter(s)")


     constant: 1 trainable parameter(s)
 running_diff: 2 trainable parameter(s)


Now we can pass a batch to each model to confirm it returns one logit per sample. Note
that our outputs have the size of our batch.

In [13]:
batch = next(iter(train_data_loader))

# RunningDifferenceLogisticModel works in signum-log space, not normalized space.
# destandardize_channels undoes the per-channel z-score but KEEPS the log compression —
# raw DN values span too many orders of magnitude to be good features for one linear layer.
# (For true physical units, use train_dataset.inverse_transform_data() instead. See the
#  "THE THREE SPACES" block in workshop_infrastructure/datasets/helio.py.)
batch_logspace = destandardize_channels(batch, channel_order=cfg.data.channels, scalers=scalers)

output_constant = model_constant.forward(batch)[:, 0]              # ignores 'ts' entirely
output_running_diff = model_running_diff.forward(batch_logspace)[:, 0]

print("constant baseline output (logits):", output_constant)
print("running-difference baseline output (logits):", output_running_diff)


constant baseline output (logits): tensor([0., 0.], grad_fn=<SelectBackward0>)
running-difference baseline output (logits): tensor([0.0560, 0.0552], grad_fn=<SelectBackward0>)


## Define your metrics

Metrics are a very important part of training AI models. They provide your models with
the quantitification of error, which in turn shifts the weights towards better
pefrorming models. They also provide a way for you to monitor performance, identify
overfitting, and quantify value added.

We now initialize the metrics class which allows you to control what metrics do you want
to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones
for monitoring performance. `WaveMetrics` has been converted for this task to binary
classification: BCE-with-logits for the loss, accuracy + AUROC for reporting. As with
other components, this takes the form of a loaded module that can be later used in a
training script.

In [14]:
from downstream_apps.test.metrics.template_metrics import WaveMetrics

In [15]:
def make_metrics() -> dict:
    """Build a fresh dict of WaveMetrics instances.

    Each of the two baselines below needs its own dict: the accuracy/AUROC torchmetrics
    objects inside WaveMetrics accumulate internal state across calls, so sharing one
    instance between two independently-trained models would mix their statistics.
    """
    return {
        'train_loss': WaveMetrics("train_loss"),
        # val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
        # It defaults to the same BCE as train_loss — override WaveMetrics.val_loss to change it.
        'val_loss': WaveMetrics("val_loss"),
        'train_metrics': WaveMetrics("train_metrics"),
        # Reported only: val_metrics do NOT influence checkpoint selection.
        'val_metrics': WaveMetrics("val_metrics"),
    }

demo_metrics = make_metrics()

Now they can be evaluated on a model's output and our ground truth. First the loss that
actually will backpropagate — binary cross-entropy with logits — demonstrated here on the
running-difference baseline's output.

In [16]:
demo_metrics["train_loss"](output_running_diff, batch["forecast"])

({'bce': tensor(0.6657, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)}, [1])

Then a training evaluation that will not backpropagate and inform our model, but that we
can keep an eye on. Note that reporting lots of metrics during training will slow the
training process. I'm including it here as an example, but oftentimes it is better to put
the diagnostics only in the validation evaluation metrics.

Here we are calculating accuracy (fraction correctly classified at a 0.5 probability
threshold). Since torchmetrics' binary metrics auto-apply sigmoid to logit inputs, this
works directly on the raw model output.

In [17]:
demo_metrics["train_metrics"](output_running_diff, batch["forecast"])

({'accuracy': tensor(0.)}, [1])

In the validation evaluation metrics we report both accuracy and AUROC.

In [18]:
demo_metrics["val_metrics"](output_running_diff, batch["forecast"])

/home/jovyan/envs/surya_WS/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)


({'accuracy': tensor(0.), 'auroc': tensor(0.)}, [1, 1])

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the WaveLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

In [19]:
from downstream_apps.test.lightning_modules.pl_simple_baseline import WaveLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [20]:
L.seed_everything(42, workers=True)

Seed set to 42


42

## Intialize Lightning modules

Now we properly initialize one Lightning module per baseline, each with its own metrics
dict (see `make_metrics()` above) so their statistics don't mix.

In [21]:
from functools import partial

# ---------------------------------------------------------------------------
# Learning rate. NOT cfg.learning_rate.
#
# cfg.learning_rate is 0.01, which was tuned for the earlier 16,385-parameter version of
# this baseline. It is far too small for the 2-parameter model. The feature
# mean(|pool32(now - prev)|) is of order 0.1, so separating the classes needs a weight of
# order 10-100. Adam moves a parameter by roughly `lr` per step, so at lr=0.01 that is
# ~3000 steps of pure travel before the model is even in the right range, and the run ends
# looking flat and uninformative.
#
# 0.1 reaches weight ~30 in ~300 steps — under two epochs at 192 steps/epoch — and leaves
# the residual oscillation at ~0.3% of the weight, so the calibration (and therefore the
# loss) settles cleanly. If you cut TRAIN_N so far that an epoch is only a few steps, raise
# this to 1.0; the trade is convergence speed against how tightly the loss can settle.
#
# This only affects loss and calibration. AUROC is a property of the feature itself, so it
# is unchanged by the learning rate — see the diagnostic cell at the end.
# cfg.learning_rate is deliberately left alone, for Surya in notebook 2.
# ---------------------------------------------------------------------------
baseline_lr = 0.1
print(f"baseline_lr = {baseline_lr} (config's learning_rate = {cfg.learning_rate}, "
      f"kept for Surya)")

# Baseline 1: constant probability — never touches 'ts', so no preprocess_fn needed.
lit_model_constant = WaveLightningModule(
    model_constant, make_metrics(), lr=baseline_lr, batch_size=batch_size,
)

# Baseline 2: running difference — needs the signum-log-space inverse transform wired up
# so WaveLightningModule applies it before every model call.
preprocess_fn = partial(
    destandardize_channels,
    channel_order=cfg.data.channels,
    scalers=scalers,
)
lit_model_running_diff = WaveLightningModule(
    model_running_diff, make_metrics(), lr=baseline_lr, batch_size=batch_size,
    preprocess_fn=preprocess_fn,
)


baseline_lr = 0.1 (config's learning_rate = 0.01, kept for Surya)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [25]:
# WandB is now authenticated via `wandb login` (stored in ~/.netrc as lloverasdiego /
# surya-ws2), so WandbLogger below logs live — no offline/disabled override needed.

In [22]:
project_name = cfg.wandb_project

# Figures are written here as PNG *and* attached to the wandb run, so the same numbers are
# available offline and online.
RESULTS_DIR = Path("experiments/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def make_loggers(run_name: str) -> list:
    """Build a fresh WandB + CSV logger pair for one baseline's run.

    The `wandb.finish()` at the top is the important part. Lightning *reuses* an
    in-progress wandb run when a new WandbLogger is constructed — it warns "There is a
    wandb run already in progress and newly created instances of `WandbLogger` will reuse
    this run". That is how the two baselines previously ended up sharing one run: both
    logged `val_loss` at the same step numbers, so the second silently overwrote the first
    and neither curve could be read. Ending the run here means each baseline gets its own.
    """
    if wandb.run is not None:
        wandb.finish()

    wandb_logger = WandbLogger(
        entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
        project=project_name,
        name=run_name,
        log_model=False,
        save_dir="./wandb/wandb_tmp",
    )
    csv_logger = CSVLogger("runs", name=f"{project_name}_{run_name}")
    return [wandb_logger, csv_logger]


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).

In [23]:
#max_epochs = 4  # previous value: hardcoded, ignored cfg.max_epochs entirely
max_epochs = cfg.max_epochs  # now driven by training.max_epochs in config_script.yaml


def make_trainer(run_name: str) -> L.Trainer:
    """One Trainer for one baseline, built immediately before its own `.fit()`.

    This used to construct both Trainers in a single cell. That is the bug behind the
    shared wandb run: constructing a Trainer constructs its WandbLogger, so two Trainers
    built back to back opened one run and reused it for the second. Building the Trainer
    inside a factory, called from the fit cell, means `make_loggers()` runs at the moment
    the previous run has already been finished.
    """
    return L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices="auto",
        logger=make_loggers(run_name),
        # val_loss is the monitored quantity; save_top_k=1 keeps the single best epoch.
        callbacks=[ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)],
        # Lightning logs training metrics every N steps but validation metrics once per
        # epoch. Keep N well below the steps-per-epoch count (TRAIN_N // batch_size) or
        # train_loss lands roughly once per epoch and the training curve looks like noise.
        log_every_n_steps=5,
    )


def finish_run(run_name: str, trainer: L.Trainer) -> list:
    """Plot the per-epoch curves for a finished run, save them, attach them to wandb, end it.

    The figures are built from the CSV logger's `metrics.csv` — the *same* values Lightning
    streamed to wandb — so the local PNG and the wandb dashboard cannot disagree. The
    plotting itself is `wave_common.plot_run()`, shared with 4_finetune_wave_1D.py rather
    than reimplemented here.
    """
    from IPython.display import Image as IPyImage, display

    csv_dir = Path(trainer.loggers[1].log_dir)
    written = wc.plot_run(run_name, csv_dir, evals={}, out_dir=RESULTS_DIR)

    for path in written:
        if wandb.run is not None:
            wandb.log({f"figures/{path.stem}": wandb.Image(str(path))})
        print(f"wrote {path}")
        display(IPyImage(filename=str(path)))

    if wandb.run is not None:
        wandb.finish()
    return written


## Fit the models

Finally we fit each baseline. We pass its Lightning module, and the shared dataloaders
(both baselines see the same train/val split, so their validation performance is directly
comparable).

In [24]:
#trainer_constant.fit(lit_model_constant, train_data_loader, val_data_loader)
# ^ previous version: commented out, so the base-rate reference was never actually trained
#   and the running-difference baseline had nothing to be compared against.

#run_name_constant = "baseline_constant_prob"
#trainer_constant = make_trainer(run_name_constant)
#trainer_constant.fit(lit_model_constant, train_data_loader, val_data_loader)

# Expected: val_loss settles at ln 2 = 0.6931 and accuracy at 0.5. The splits are exactly
# class-balanced by construction, so the best a model that ignores the image can do is
# predict p=0.5 — this run measures the floor everything else has to beat.
#figs_constant = finish_run(run_name_constant, trainer_constant)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name  | Type                     | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | model | ConstantProbabilityModel | 1      | train | 0    
-------------------------------------------------------------------
1         Trainable params
0         Non-trainable params
1         Total params
0.000     Total estimated model params size (MB)
1         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [25]:
# Safety net, and safe to re-run. `finish_run()` above already ends the run, so normally
# this reports "no live wandb run" — it is here for the case where a fit raised partway
# through and left one open, which would otherwise be silently reused by the next baseline.
#
# Previously this cell was a bare `wandb.finish()` and raised
#   NameError: name 'wandb' is not defined
# because cell 3 imported only Lightning's WandbLogger wrapper, never the wandb module.
if wandb.run is not None:
    print(f"finishing live wandb run: {wandb.run.name}")
    wandb.finish()
else:
    print("no live wandb run")


finishing live wandb run: baseline_constant_prob


In [30]:
# This assignment is a no-op when the config already carries the right entity (it does:
# logging.wandb_entity in config_script.yaml). It is kept because it is the one knob to turn
# if you are running under a different wandb account.
#
# What matters is WHERE it runs. cfg.wandb_entity is read inside make_loggers(), at the
# moment a Trainer is built — so this must execute BEFORE the fit cell whose run it should
# affect. In the previous version of this notebook it ran *after* both Trainers had already
# been constructed, so it changed nothing about the runs that existed.
cfg.wandb_entity = "lloverasdiego-grupo-de-estudios-en-heliofisica-de-mendoza"
print(f"wandb_entity now: {cfg.wandb_entity} | project: {project_name}")


wandb_entity now: lloverasdiego-grupo-de-estudios-en-heliofisica-de-mendoza | project: wave_classification


In [27]:
run_name_running_diff = "baseline_running_diff"
trainer_running_diff = make_trainer(run_name_running_diff)
trainer_running_diff.fit(lit_model_running_diff, train_data_loader, val_data_loader)

# The result to read: does val_loss go BELOW the constant baseline's ln 2 = 0.6931?
# With two parameters the model cannot memorize the training set, so unlike the earlier
# 16,385-parameter version there is no overfitting story available — train and val loss
# should move together. If val_loss stalls at ln 2 while AUROC is above 0.5, the feature
# ranks the classes correctly but the calibration has not converged: raise baseline_lr.
figs_running_diff = finish_run(run_name_running_diff, trainer_running_diff)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name  | Type                           | Params | Mode  | FLOPs
-------------------------------------------------------------------------
0 | model | RunningDifferenceLogisticModel | 2      | train | 0    
-------------------------------------------------------------------------
2         Trainable params
0         Non-trainable params
2         Total params
0.000     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jovyan/envs/surya_WS/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


/home/jovyan/envs/surya_WS/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

## Diagnostic: is the feature informative at all?

The two runs above tell you what the models *learned*. For a two-parameter model that is
monotone in a single scalar, that is less than it sounds: the ordering it induces on the
validation samples is fixed the moment the weight has a sign, so **AUROC is a property of
the feature, not of the training**. Only the loss and the threshold are learned.

So the loss curves cannot answer the question that matters — *does
`mean(|pool32(AIA193 now − prev)|)` separate wave from no-wave at all?* The cell below
answers it directly, with no training, on the validation split.


In [ ]:
# ---------------------------------------------------------------------------
# What the single feature actually separates — no training involved.
#
# The running-difference model is monotone in one scalar, so ANY positive weight produces
# the same ranking of validation samples. AUROC is therefore a property of the FEATURE,
# fixed before the first gradient step; training moves only the loss and the decision
# threshold. That is why the loss curves above cannot tell you whether the feature is
# informative, and this cell can.
#
# Read it as: if AUROC here is ~0.5, the feature carries no class information and no
# learning rate or epoch count will rescue the baseline — that is a real result about
# AIA193 running differences at this label timing, not a bug. If AUROC is well above 0.5
# but val_loss sat at ln 2, the ranking is fine and only the calibration failed to
# converge, which baseline_lr fixes.
# ---------------------------------------------------------------------------
from IPython.display import Image as IPyImage, display
from sklearn.metrics import roc_auc_score, roc_curve

features, labels = [], []
for val_batch in val_data_loader:
    labels.append(val_batch["forecast"].float().reshape(-1).numpy())
    # Same two transforms the trained model sees, in the same order: undo the z-score into
    # signum-log space, then compute the pooled magnitude. Using the model's own feature
    # function (imported above) rather than a copy is what keeps this honest.
    val_logspace = destandardize_channels(
        val_batch, channel_order=cfg.data.channels, scalers=scalers
    )
    feature = mean_abs_pooled_running_difference(
        val_logspace["ts"], aia193_index, POOL_KERNEL
    )
    features.append(feature.detach().reshape(-1).numpy())

features = np.concatenate(features)
labels = np.concatenate(labels)

auroc = roc_auc_score(labels, features)
mean_wave = float(features[labels == 1].mean())
mean_no_wave = float(features[labels == 0].mean())
print(f"val samples: {labels.size} "
      f"({int(labels.sum())} wave, {int((1 - labels).sum())} no wave)")
print(f"feature mean — wave: {mean_wave:.4f} | no wave: {mean_no_wave:.4f}")
print(f"feature AUROC: {auroc:.4f}   (0.5 = carries no class information)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].hist(features[labels == 0], bins=15, alpha=0.6, label="no wave")
axes[0].hist(features[labels == 1], bins=15, alpha=0.6, label="wave")
axes[0].set(xlabel=f"mean(|pool{POOL_KERNEL}(AIA193 now - prev)|)", ylabel="count",
            title="validation feature distribution")
axes[0].legend(fontsize=8)

fpr, tpr, _ = roc_curve(labels, features)
axes[1].plot(fpr, tpr, label=f"AUROC {auroc:.3f}")
axes[1].plot([0, 1], [0, 1], ":", c="0.5", label="chance")
axes[1].set(xlabel="false positive rate", ylabel="true positive rate",
            title="feature ROC (untrained)")
axes[1].legend(fontsize=8)

diag_path = RESULTS_DIR / "baseline_feature_diagnostic.png"
fig.savefig(diag_path, dpi=130)
plt.close(fig)
print(f"wrote {diag_path}")
# Displayed from the PNG rather than shown inline: plot_run() switches matplotlib to the
# headless Agg backend (it has to, for the training script), so plt.show() would be a no-op.
display(IPyImage(filename=str(diag_path)))

# Its own wandb run, so the figure sits beside the two training runs instead of being
# appended to whichever one happened to still be open.
wandb.init(entity=cfg.wandb_entity, project=project_name,
           name="baseline_feature_diagnostic", dir="./wandb/wandb_tmp")
wandb.log({
    "feature/auroc": auroc,
    "feature/mean_wave": mean_wave,
    "feature/mean_no_wave": mean_no_wave,
    "figures/feature_diagnostic": wandb.Image(str(diag_path)),
})
wandb.finish()


## Conclusion

We now have two trainable baselines — constant probability (1 parameter) and AIA193 running
difference (2 parameters) — trained and scored on the **same** train/val split, each in its
own WandB run, with per-epoch curves saved both to `experiments/results/` and attached to
the run as media.

What to compare, in order:

1. **`val_loss` against ln 2 = 0.6931.** The constant baseline defines that floor: the
   splits are exactly class-balanced, so a model that ignores the image can do no better.
   Anything the running-difference baseline gains has to show up as `val_loss` below it.
2. **`val_metric_auroc` against 0.5**, cross-checked against the untrained feature AUROC in
   the diagnostic cell. These two numbers should agree closely — if the trained run's AUROC
   is much worse, the weight has converged to the wrong sign.
3. **`train_loss` against `val_loss`.** With two parameters there is no capacity to
   memorize, so they should move together. A gap here would mean the two splits differ in
   distribution, not that the model overfit.

The next step is notebook 2: substitute Surya for these baselines, on the identical events
(same index files, same `SUBSET_SEED`, same `wave_common.build_wave_datasets()` call) and
see whether 1.5 M LoRA parameters beat two.
